# Análise de Furtos e Roubos na Grande Vitória

**Disciplina:** Análise de Dados — Projeto Integrador III  

**Grupo:**  Alexsander, Ester, Larissa Moraes, Lucas Rufino, Marcelo Mindas, Vanderson de Almeida.

**Data:** Maio de 2026 


---

## 1. Introdução

Este notebook apresenta uma análise exploratória dos dados de furtos e roubos
registrados na região da Grande Vitória (ES), obtidos no portal [SESP](https://sesp.es.gov.br/painel-de-crimes-contra-o-patrimonio)

O objetivo é identificar padrões temporais, geográficos e por tipo de ocorrência,
respondendo às seguintes perguntas:

1. Qual município concentra mais ocorrências?
2. Como os crimes se distribuem ao longo dos meses?
3. Em quais horários as ocorrências são mais frequentes?
4. Existe diferença no padrão entre furtos e roubos?

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
dados = pd.read_csv("../dados/brutos/MICRODADOS_OCORRENCIAS.csv", sep=';', encoding='latin-1')
dados.sample(10, random_state=42)

,Data,Hora,TipoIncidente,TipoLocal,Municipio,Bairro
305481,16/06/2022,18:52:00,ROUBO: A PESSOA EM VIA PÚBLICA,VIA PÚBLICA,VILA VELHA,DIVINO ESPIRITO SANTO
410545,08/02/2021,15:12:00,ROUBO: EM ESTABELECIMENTO COMERCIAL,COMÉRCIO,SERRA,PARQUE RESIDENCIAL LARANJEIRAS
383403,06/07/2021,Indeterminada,ESTELIONATO/FRAUDE,AMBIENTE WEB,VITORIA,JUCUTUQUARA
313304,13/05/2022,22:00:00,ROUBO: A PESSOA EM VIA PÚBLICA,VIA PÚBLICA,CARIACICA,CAMPO GRANDE
185279,12/12/2023,Indeterminada,ESTELIONATO/FRAUDE,AMBIENTE WEB,GUARAPARI,OUTRO LOCAL
296597,27/07/2022,00:00:00,ESTELIONATO/FRAUDE,NaN,SERRA,JARDIM LIMOEIRO
190848,17/11/2023,15:30:00,ESTELIONATO/FRAUDE,AMBIENTE WEB,MIMOSO DO SUL,SANTA ROSA
288033,03/09/2022,18:00:00,ROUBO: A PESSOA EM VIA PÚBLICA,VIA PÚBLICA,CARIACICA,CAMPO GRANDE
361819,21/10/2021,17:30:00,ROUBO: A PESSOA EM VIA PÚBLICA,VIA PÚBLICA,CARIACICA,CAMPO GRANDE
486620,17/11/2019,16:40:00,ROUBO: A PESSOA EM VIA PÚBLICA,VIA PÚBLICA,VIANA,CANAA


In [3]:
dados.columns

Index(['Data', 'Hora', 'TipoIncidente', 'TipoLocal', 'Municipio', 'Bairro'], dtype='str')

In [4]:
dados.shape

(617940, 6)

In [5]:
dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 617940 entries, 0 to 617939
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   Data           617940 non-null  str  
 1   Hora           617940 non-null  str  
 2   TipoIncidente  617940 non-null  str  
 3   TipoLocal      535736 non-null  str  
 4   Municipio      617940 non-null  str  
 5   Bairro         617940 non-null  str  
dtypes: str(6)
memory usage: 28.3 MB


In [6]:
dados.isnull().sum()

Data                 0
Hora                 0
TipoIncidente        0
TipoLocal        82204
Municipio            0
Bairro               0
dtype: int64

## Filtragem inicial da base de dados

In [7]:
# Padronização de letras maiúsculas/minúsculas e espaços
dados["TipoIncidente"] = dados["TipoIncidente"].str.upper().str.strip()
dados["Municipio"] = dados["Municipio"].str.upper().str.strip()
dados["Bairro"] = dados["Bairro"].str.upper().str.strip()
dados["TipoLocal"] = dados["TipoLocal"].str.upper().str.strip()

In [8]:
# Conversão de datas
dados["Data"] = pd.to_datetime(dados["Data"], errors="coerce")

print("Período original da base:")
print(dados["Data"].min(), "até", dados["Data"].max())

Período original da base:
2018-01-01 00:00:00 até 2026-04-30 00:00:00


C:\Users\ESTER\AppData\Local\Temp\ipykernel_32920\3350969087.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dados["Data"] = pd.to_datetime(dados["Data"], errors="coerce")


In [9]:
# Período de tempo 
dt_inicio = pd.Timestamp("2021-03-01")
dt_fim = pd.Timestamp("2026-03-30")

dados_filtr = dados[
    (dados["Data"] >= dt_inicio) &
    (dados["Data"] <= dt_fim)
].copy()

In [10]:
# Filtragem dos Municípios
municipios_gv = [
    "VITORIA",
    "VILA VELHA",
    "SERRA",
    "CARIACICA",
    "VIANA",
    "GUARAPARI",
    "FUNDAO"
]
dados_filtr = dados_filtr[
    dados_filtr["TipoIncidente"].isin(municipios_gv)
].copy()

In [11]:
# Filtragem de Furtos e Roubos
dados_filtr = dados_filtr[
    dados_filtr ["TipoIncidente"].str.contains("FURTO|ROUBO", na=False)
].copy()

In [12]:
# Categoria resumida
dados_filtr["CategoriaCrime"] = dados_filtr["TipoIncidente"].apply(
    lambda tipo: "Furto" if "FURTO" in tipo else "Roubo"
)

In [13]:
# Variáveis de tempo
dados_filtr["Ano"] = dados_filtr["Data"].dt.year
dados_filtr["Mes"] = dados_filtr["Data"].dt.month
dados_filtr["AnoMes"] = dados_filtr["Data"].dt.to_period("M")
dados_filtr["DiaSemana"] = dados_filtr["Data"].dt.day_name()

traducao_dias = {
    "Monday": "Segunda-feira",
    "Tuesday": "Terça-feira",
    "Wednesday": "Quarta-feira",
    "Thursday": "Quinta-feira",
    "Friday": "Sexta-feira",
    "Saturday": "Sábado",
    "Sunday": "Domingo"
}

dados_filtr["DiaSemana"] = dados_filtr["DiaSemana"].map(traducao_dias)


In [14]:
# Tratamento de Hora
dados_filtr["HoraConvertida"] = pd.to_datetime(
    dados_filtr["Hora"],
    format="%H:%M:%S",
    errors="coerce"
)

dados_filtr["HoraDia"] = dados_filtr["HoraConvertida"].dt.hour

In [15]:
dados_hora = dados_filtr.dropna(subset=["HoraDia"]).copy()

# TESTES

In [16]:
print("Tamanho da base original:", dados.shape)
print("Tipo da coluna Data:", dados["Data"].dtype)

print("\nPrimeiras datas:")
display(dados["Data"].head(10))

print("\nMenor e maior data na base original:")
print(dados["Data"].min(), dados["Data"].max())

Tamanho da base original: (617940, 6)
Tipo da coluna Data: datetime64[us]

Primeiras datas:


0   2026-04-30
1   2026-04-30
2   2026-04-30
3   2026-04-30
4   2026-04-30
5   2026-04-30
6   2026-04-30
7   2026-04-30
8   2026-04-30
9   2026-04-30
Name: Data, dtype: datetime64[us]


Menor e maior data na base original:
2018-01-01 00:00:00 2026-04-30 00:00:00


In [17]:
dados["DataConvertida"] = pd.to_datetime(dados["Data"], errors="coerce")

print("Datas que não foram convertidas:")
print(dados["DataConvertida"].isna().sum())

print("\nMenor e maior data convertida:")
print(dados["DataConvertida"].min(), dados["DataConvertida"].max())

Datas que não foram convertidas:
0

Menor e maior data convertida:
2018-01-01 00:00:00 2026-04-30 00:00:00


In [18]:
# Conferência da base original
print("Base original:", dados.shape)
print("Período original:", dados["Data"].min(), "até", dados["Data"].max())

# Filtro de período definido no projeto
dt_inicio = pd.Timestamp("2021-03-01")
dt_fim = pd.Timestamp("2026-04-30")

dados_periodo = dados[
    (dados["Data"] >= dt_inicio) &
    (dados["Data"] <= dt_fim)
].copy()

print("Após filtro de período:", dados_periodo.shape)
print("Período filtrado:", dados_periodo["Data"].min(), "até", dados_periodo["Data"].max())

Base original: (617940, 7)
Período original: 2018-01-01 00:00:00 até 2026-04-30 00:00:00
Após filtro de período: (406806, 7)
Período filtrado: 2021-03-01 00:00:00 até 2026-04-30 00:00:00


In [19]:
dados_periodo["Municipio"].value_counts().head(30)

Municipio
VILA VELHA                 71702
SERRA                      70890
VITORIA                    63471
CARIACICA                  45620
CACHOEIRO DE ITAPEMIRIM    16830
LINHARES                   16657
GUARAPARI                  14268
SAO MATEUS                 11348
COLATINA                   10360
ARACRUZ                     7660
VIANA                       6001
MARATAIZES                  4860
NOVA VENECIA                3807
ITAPEMIRIM                  3007
BARRA DE SAO FRANCISCO      2665
ANCHIETA                    2512
SAO GABRIEL DA PALHA        2498
PIUMA                       2242
CASTELO                     2185
GUACUI                      2040
SANTA MARIA DE JETIBA       1893
BAIXO GUANDU                1856
CONCEICAO DA BARRA          1835
IUNA                        1717
VENDA NOVA DO IMIGRANTE     1686
JAGUARE                     1580
FUNDAO                      1502
PINHEIROS                   1433
SOORETAMA                   1422
ALEGRE                      1356


In [20]:
dados_periodo[
    dados_periodo["Municipio"].str.contains("VIANA|FUNDAO", na=False)
]["Municipio"].value_counts()

Municipio
VIANA     6001
FUNDAO    1502
Name: count, dtype: int64

In [21]:
municipios_gv = [
    "VITORIA",
    "VILA VELHA",
    "SERRA",
    "CARIACICA",
    "VIANA",
    "GUARAPARI",
    "FUNDAO"
]

dados_gv = dados_periodo[
    dados_periodo["Municipio"].isin(municipios_gv)
].copy()

print("Após filtro de municípios:", dados_gv.shape)

display(dados_gv["Municipio"].value_counts())

Após filtro de municípios: (273454, 7)


Municipio
VILA VELHA    71702
SERRA         70890
VITORIA       63471
CARIACICA     45620
GUARAPARI     14268
VIANA          6001
FUNDAO         1502
Name: count, dtype: int64

In [22]:
dados_gv["TipoIncidente"].value_counts().head(30)

TipoIncidente
ESTELIONATO/FRAUDE                     144364
ROUBO: A PESSOA EM VIA PÚBLICA          59092
FURTO: EM ESTABELECIMENTO COMERCIAL     15016
FURTO: EM RESIDÊNCIA/CONDOMÍNIO         12864
FURTO: A PESSOA EM VIA PÚBLICA          12568
FURTO: EM TRANSPORTE COLETIVO            9067
CRIMES INFORMÁTICOS                      8394
ROUBO: EM TRANSPORTE COLETIVO            8236
ROUBO: EM ESTABELECIMENTO COMERCIAL      3155
ROUBO: EM RESIDÊNCIA/CONDOMÍNIO           698
Name: count, dtype: int64

In [23]:
dados_crimes = dados_gv[
    dados_gv["TipoIncidente"].str.contains("FURTO|ROUBO", na=False)
].copy()

print("Após filtro de furtos e roubos:", dados_crimes.shape)

display(dados_crimes["TipoIncidente"].value_counts().head(30))

Após filtro de furtos e roubos: (120696, 7)


TipoIncidente
ROUBO: A PESSOA EM VIA PÚBLICA         59092
FURTO: EM ESTABELECIMENTO COMERCIAL    15016
FURTO: EM RESIDÊNCIA/CONDOMÍNIO        12864
FURTO: A PESSOA EM VIA PÚBLICA         12568
FURTO: EM TRANSPORTE COLETIVO           9067
ROUBO: EM TRANSPORTE COLETIVO           8236
ROUBO: EM ESTABELECIMENTO COMERCIAL     3155
ROUBO: EM RESIDÊNCIA/CONDOMÍNIO          698
Name: count, dtype: int64

In [24]:
dados_crimes["CategoriaCrime"] = dados_crimes["TipoIncidente"].apply(
    lambda tipo: "Furto" if "FURTO" in tipo else "Roubo"
)

display(dados_crimes["CategoriaCrime"].value_counts())

CategoriaCrime
Roubo    71181
Furto    49515
Name: count, dtype: int64